In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
from tqdm import tqdm 
import math
from video2tensor import *
import shutil
import os 

In [14]:
def scrape_videos(start,
                  end,
                  driver,
                  num_in_batch,
                  videos_folder,
                  target_fps,
                  batch_size,
                  ft_folder,
                  embedding):
    
    press_times = math.ceil(end/50)
    
    for i in range(press_times):
        ele  = driver.find_element(By.XPATH,'//button[@class = "flex grow items-center justify-center group-hover:underline"]')
        ele.click()
        time.sleep(5)

    eles  = driver.find_elements(By.XPATH,'//a[@class = "group col-span-4 flex items-center justify-self-end truncate text-right font-mono text-[0.8rem] leading-6 text-gray-400 md:col-span-3 lg:col-span-2 xl:pr-10"]')

    len_eles = len(eles)
    if end>len_eles:
        raise ValueError(f'total numbers of videos in the current page:{len}')

    init,cnt,target_len = 0,0,end-start+1
    is2process = False
    while cnt<target_len:
        if cnt//num_in_batch <= init:
            try:
                eles[start+cnt].click()
                time.sleep(2)
                cnt +=1
            except Exception as e:
                print(f'An error occured in {start} row.')
        else:
            is2process = True
            
        if cnt== target_len or is2process:
            print(f'New Batch created. By now, {cnt} videos have been downloaded.')
            print("Transfer videos to videos folder")
            
            if not os.path.isdir(videos_folder):
                os.makedirs(videos_folder)
                
            grant1 = 'n' 
            while grant1 == 'n':
                grant1 = input('Transfer completed?[y/n]?')
            
            video2tensor(videos_folder=videos_folder,
                         target_fps=target_fps,
                         batch_size=batch_size,
                         ft_folder=ft_folder,
                         embedding=embedding)
            
            print('Embedding for this batch is completed. \
                   Transfer the embedding to google cloud and save videos_basic_info for future use.')
            
            grant2 = 'n' 
            while grant2 == 'n':
                grant2 = input('Transfer completed?[y/n]?')
            
            shutil.rmtree(videos_folder)
            shutil.rmtree(ft_folder)
            
            init +=1
            is2process = False
    print('All required videos have been successfully processed.')

In [15]:
#set parmas for video2tensor
abspath = os.path.join(os.getcwd(),'..')
videos_folder = os.path.join(abspath,'videos')
ft_folder = os.path.join(abspath,'features')
target_fps = 30
embedding = ImgEmbedding()
embedding.set_model(model_name="test")
batch_size = (192,192)

/opt/anaconda3/envs/py3.12.3/lib/python3.12/site-packages/timm/models/_factory.py:126: UserWarning: Mapping deprecated model name swinv2_base_window12_192_22k to current swinv2_base_window12_192.ms_in22k.
  model = create_fn(


In [16]:
#right closed, left closed
start,end = 101,130
# It's index fule. For example, in list [0], to retrieve '0', set start = 0,end = 0.

chromeOptions = webdriver.ChromeOptions()
chromeOptions.add_experimental_option("excludeSwitches", ['enable-automation'])
hg_url = 'https://huggingface.co/datasets/shuaishuaicdp/GUI-World/tree/main/website'
driver = webdriver.Chrome(chromeOptions)
driver.get(hg_url)
scrape_videos(start,end,driver=driver,num_in_batch=25,videos_folder=videos_folder,target_fps=target_fps,
              batch_size=batch_size,ft_folder=ft_folder,embedding=embedding)
driver.quit()

New Batch created. By now, 25 videos have been downloaded.
Transfer videos to videos folder


100%|██████████| 26/26 [07:37<00:00, 17.59s/it]


Embedding for this batch is completed. Transfer the embedding to google cloud.
New Batch created. By now, 30 videos have been downloaded.
Transfer videos to videos folder


0it [00:00, ?it/s]


Embedding for this batch is completed. Transfer the embedding to google cloud.
All required videos have been successfully processed.
